In [1]:
import pandas as pd
import sqlalchemy
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path


In [2]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [3]:
# showing all the model metrics stored in the database with comparison of accuracy, precision, recall, f1 score and roc auc of all the models
query = """
SELECT *
FROM model_metrics
ORDER BY roc_auc DESC
"""

model_metrics = pd.read_sql(
    query,
    engine
)

model_metrics

,model_name,accuracy,precision,recall,f1_score,roc_auc,run_date
0,XGBoost,0.991988,0.953157,0.781302,0.858716,0.993212,2026-08-10 01:41:16.192393
1,RANDOM FOREST,0.991571,1.000000,0.729549,0.843629,0.989129,2026-08-10 01:40:45.529107
2,Logistic Regression,0.952549,0.388612,0.911519,0.544910,0.981138,2026-08-10 01:39:59.312970


In [4]:
# showing teh bestt model based on the highest roc auc score
best_model = model_metrics.iloc[0]

print("Best Model:")
print(best_model)

Best Model:
model_name                       XGBoost
accuracy                        0.991988
precision                       0.953157
recall                          0.781302
f1_score                        0.858716
roc_auc                         0.993212
run_date      2026-08-10 01:41:16.192393
Name: 0, dtype: object


In [5]:
# ranking the models based on their roc auc score in descending order
model_metrics["rank"] = (
    model_metrics["roc_auc"]
    .rank(
        ascending=False
    )
)

model_metrics.sort_values(
    "rank"
)

,model_name,accuracy,precision,recall,f1_score,roc_auc,run_date,rank
0,XGBoost,0.991988,0.953157,0.781302,0.858716,0.993212,2026-08-10 01:41:16.192393,1.0
1,RANDOM FOREST,0.991571,1.000000,0.729549,0.843629,0.989129,2026-08-10 01:40:45.529107,2.0
2,Logistic Regression,0.952549,0.388612,0.911519,0.544910,0.981138,2026-08-10 01:39:59.312970,3.0


In [6]:
import pandas as pd

from sqlalchemy import create_engine

DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

metrics_df = pd.read_sql(
    """
    SELECT *
    FROM model_metrics
    """,
    engine
)

registry_df = metrics_df.copy()

# Create version column
registry_df["version"] = "v1"

registry_df = registry_df[
    [
        "model_name",
        "version",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc",
        "run_date"
    ]
]

registry_df.columns = [
    "model_name",
    "version",
    "accuracy",
    "precision_score",
    "recall_score",
    "f1_score",
    "roc_auc",
    "trained_at"
]

print(registry_df.head())

registry_df.to_sql(
    "model_registry",
    con=engine,
    if_exists="replace",
    index=False
)

print("Model Registry Created Successfully")

            model_name version  accuracy  precision_score  recall_score  \
0  Logistic Regression      v1  0.952549         0.388612      0.911519   
1        RANDOM FOREST      v1  0.991571         1.000000      0.729549   
2              XGBoost      v1  0.991988         0.953157      0.781302   

   f1_score   roc_auc                 trained_at  
0  0.544910  0.981138 2026-08-10 01:39:59.312970  
1  0.843629  0.989129 2026-08-10 01:40:45.529107  
2  0.858716  0.993212 2026-08-10 01:41:16.192393  


Model Registry Created Successfully
